In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:

import pandas as pd
import numpy as np
import csv
import pickle as pkl
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
import os




In [3]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "LLM4BEAR Dataset"

Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 61 (delta 6), reused 38 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 37.75 KiB | 2.90 MiB/s, done.
Resolving deltas: 100% (6/6), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 82 (delta 7), reused 81 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 46.13 MiB | 22.45 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (83/83), done.
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 10

In [4]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

In [5]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])


electronic_items = [electronic_metadata.iloc[i]['titles'] for i in range(len(electronic_metadata))]

food_items = [food_metadata.iloc[i]['titles'] for i in range(len(food_metadata))]

clothing_items = [clothing_metadata.iloc[i]['titles'] for i in range(len(clothing_metadata))]

This device is a plug-and-play USB adapter that provides 5.1 channel surround sound capabilities to computers without the need for an internal sound card.
This is a pink swimsuit designed for girls aged 7 to 16, featuring a sweetheart neckline and suitable for swimming and beach activities.
A fragrant blend of star anise, cloves, Chinese cinnamon, Sichuan peppercorns, and ginger, this seasoning enhances a variety of dishes with its unique sweet and savory flavor profile.


In [6]:
import pickle

# --- ELECTRONIC ---
# 1. Load bundles
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_electronic_bundles.pkl", "rb") as f:
    llm4bear_electronic_bundles = pickle.load(f)

# 2. Load intents
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_electronic_intents.pkl", "rb") as f:
    llm4bear_electronic_intents = pickle.load(f)


# --- CLOTHING ---
# 1. Load bundles
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_clothing_bundles.pkl", "rb") as f:
    llm4bear_clothing_bundles = pickle.load(f)

# 2. Load intents
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_clothing_intents.pkl", "rb") as f:
    llm4bear_clothing_intents = pickle.load(f)


# --- FOOD ---
# 1. Load bundles
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_food_bundles.pkl", "rb") as f:
    llm4bear_food_bundles = pickle.load(f)

# 2. Load intents
with open("/content/LLM4BEAR/LLM4BEAR Dataset/llm4bear_food_intents.pkl", "rb") as f:
    llm4bear_food_intents = pickle.load(f)

# --- VERIFICATION ---
print("All datasets loaded successfully.")
print(f"Electronic Bundles Loaded: {len(llm4bear_electronic_bundles)}")
print(f"Clothing Bundles Loaded: {len(llm4bear_clothing_bundles)}")
print(f"Food Bundles Loaded: {len(llm4bear_food_bundles)}")

All datasets loaded successfully.
Electronic Bundles Loaded: 1750
Clothing Bundles Loaded: 1910
Food Bundles Loaded: 1784


In [13]:
def bundle_intent_maker(llm4bear_intents, domain):
    # 1. Create a dictionary with your column names and lists
    data = {
        'bundle ID': range(len(llm4bear_intents)),
        'intent': llm4bear_intents
    }

    # 2. Convert to a DataFrame
    df = pd.DataFrame(data)

    os.makedirs(f"/content/{domain}", exist_ok=True)

    # 3. Save to CSV (index=False prevents an extra column of numbers)
    df.to_csv(f'{domain}/bundle_intent.csv', index=False)

    print("CSV created successfully!")

In [31]:
bundle_intent_maker(llm4bear_electronic_intents, "electronic")
bundle_intent_maker(llm4bear_clothing_intents, "clothing")
bundle_intent_maker(llm4bear_food_intents, "food")

CSV created successfully!
CSV created successfully!
CSV created successfully!


In [27]:
electronic_bundle_item_index = []
electronic_bundle_item_ID = []

for i in range(len(llm4bear_electronic_bundles)):
    for target_title in llm4bear_electronic_bundles[i]:
        electronic_bundle_item_index.append(i)
        item_id = electronic_metadata.loc[electronic_metadata['titles'] == target_title, 'item ID'].values[0]
        electronic_bundle_item_ID.append(item_id)


clothing_bundle_item_index = []
clothing_bundle_item_ID = []

for i in range(len(llm4bear_clothing_bundles)):
    for target_title in llm4bear_clothing_bundles[i]:
        clothing_bundle_item_index.append(i)
        item_id = clothing_metadata.loc[clothing_metadata['titles'] == target_title, 'item ID'].values[0]
        clothing_bundle_item_ID.append(item_id)

food_bundle_item_index = []
food_bundle_item_ID = []

for i in range(len(llm4bear_food_bundles)):
    for target_title in llm4bear_food_bundles[i]:
        food_bundle_item_index.append(i)
        item_id = food_metadata.loc[food_metadata['titles'] == target_title, 'item ID'].values[0]
        food_bundle_item_ID.append(item_id)

In [32]:
def bundle_item_maker(bundle_item_index, bundle_item_ID, domain):
    # 1. Create a dictionary with your column names and lists
    data = {
        'bundle ID': bundle_item_index,
        'item ID': bundle_item_ID
    }

    # 2. Convert to a DataFrame
    df = pd.DataFrame(data)

    os.makedirs(f"/content/{domain}", exist_ok=True)

    # 3. Save to CSV (index=False prevents an extra column of numbers)
    df.to_csv(f'{domain}/bundle_item.csv', index=False)

    print("CSV created successfully!")

In [33]:
bundle_item_maker(electronic_bundle_item_index, electronic_bundle_item_ID, "electronic")
bundle_item_maker(clothing_bundle_item_index, clothing_bundle_item_ID, "clothing")
bundle_item_maker(food_bundle_item_index, food_bundle_item_ID, "food")

CSV created successfully!
CSV created successfully!
CSV created successfully!


In [47]:
def extract_json_from_response(response_text):
    # Find the index of the first '{' and the last '}'
    start_index = response_text.find('{')
    end_index = response_text.rfind('}')

    if start_index != -1 and end_index != -1:
        # Slice the string to get only the content between the brackets
        json_string = response_text[start_index : end_index + 1]
        return json_string
    else:
        # Return None if a valid JSON block is not found
        return None

def making_product_type_list(category_indices, all_products):
    product_list = []

    # Your original loop to gather all the products
    for i in category_indices:
        product_list = product_list + all_products[i]

    # Convert to a set to remove duplicates, then convert back to a list
    unique_product_list = list(set(product_list))

    return unique_product_list


In [48]:


electronic_category_keys = [
    'Camera and Accessories', 'Computers and Accessories', 'Audio Equipment',
    'Tablets and Accessories', 'Storage Solutions', 'Networking Equipment',
    'Mobile Devices and Accessories', 'Travel Accessories', 'Gaming',
    'Home Entertainment Systems', 'Miscellaneous Electronics', 'Power Solutions',
    'Cables and Connectors', 'Security Systems', 'Car Technology and Accessories',
    'Photography and Camera Equipment', 'Adapters and Cables',
    'Television and Accessories', 'AV Setup', 'GPS and Navigation Accessories',
    'Mobile Device Protection', 'Streaming and Media', 'PC Building and Assembly',
    'General Electronics', 'Walkie Talkies and Communication Devices'
]

clothing_category_keys = [
    'Footwear', 'Accessories', 'Costumes and Themed Apparel',
    'Lingerie and Underwear', 'Baby and Kids Clothing', 'Activewear and Sportswear',
    'Fashion Accessories', 'Seasonal and Thematic Products', 'Electronics',
    'Children\'s Items', 'Carrying Items', 'Maintenance', 'Clothing'
]

food_category_keys = [
    'Snacks', 'Beverages', 'Cooking Ingredients',
    'Breakfast Foods', 'Sweets and Desserts', 'Health Foods',
    'Canned and Packaged Foods', 'Baby Food', 'Condiments and Sauces',
    'Fruits and Vegetables', 'Specialty Foods', 'Dried and Preserved Foods',
    'Grains and Pasta', 'Miscellaneous', 'Gift Baskets and Food Gifts',
    'Dietary Specific Items', 'Cooking Tools and Kitchen Goods',
    'Sweeteners', 'Nuts and Seeds', 'Coffee/Tea',
    'Vegetables and Beans', 'Health-Conscious Options', 'Prepared and Ready-Made Meals',
    'Ethnic and Specialty Foods', 'Culinary Specialties'
]


with open(f"/content/LLM4BEAR/BundleRec Data/specific_electronic_product_metadata.pkl", 'rb') as f:
    metadata_electronic_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_clothing_product_metadata.pkl", 'rb') as f:
    metadata_clothing_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_food_product_metadata.pkl", 'rb') as f:
    metadata_food_jsons = pickle.load(f)

metadata_jsons = [metadata_clothing_jsons, metadata_electronic_jsons, metadata_food_jsons]

with open(f"/content/LLM4BEAR/BundleRec Data/general_electronic_product_list.pkl", 'rb') as f:
    general_electronic_product_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_clothing_product_list.pkl", 'rb') as f:
    general_clothing_product_list = pickle.load(f)


with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_electronic.pkl", 'rb') as f:
    all_electronic_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_electronic.pkl", 'rb') as f:
    electronic_cat_guys = pickle.load(f)

electronic_cat_guys = [extract_json_from_response(i) for i in electronic_cat_guys]

electronic_category_indices = []

for i in electronic_cat_guys:

    dictionary = json.loads(i)

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(electronic_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    electronic_category_indices.append(indices)

electronic_product_type_list = [making_product_type_list(electronic_category_indices[i], all_electronic_products) for i in range(len(electronic_cat_guys))]

with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_clothing.pkl", 'rb') as f:
    all_clothing_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_clothing.pkl", 'rb') as f:
    clothing_cat_guys = pickle.load(f)

clothing_cat_guys = [extract_json_from_response(i) for i in clothing_cat_guys]


clothing_category_indices = []

for i in clothing_cat_guys:

    dictionary = json.loads(i)

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(clothing_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    clothing_category_indices.append(indices)

clothing_product_type_list = [making_product_type_list(clothing_category_indices[i], all_clothing_products) for i in range(len(clothing_cat_guys))]


with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_food.pkl", 'rb') as f:
    all_food_products = pickle.load(f)


with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_food.pkl", 'rb') as f:
    food_cat_guys = pickle.load(f)


food_cat_guys = [extract_json_from_response(i) for i in food_cat_guys]
food_cat_guys[3255] = """{
  "Snacks": 1,
  "Beverages": 0,
  "Cooking Ingredients": 0,
  "Breakfast Foods": 0,
  "Sweets and Desserts": 0,
  "Health Foods": 0,
  "Canned and Packaged Foods": 0,
  "Baby Food": 0,
  "Condiments and Sauces": 0,
  "Fruits and Vegetables": 0,
  "Specialty Foods": 0,
  "Dried and Preserved Foods": 0,
  "Grains and Pasta": 0,
  "Miscellaneous": 0,
  "Gift Baskets and Food Gifts": 0,
  "Dietary Specific Items": 0,
  "Cooking Tools and Kitchen Goods": 0,
  "Sweeteners": 0,
  "Nuts and Seeds": 0,
  "Coffee/Tea": 0,
  "Vegetables and Beans": 0,
  "Health-Conscious Options": 0,
  "Prepared and Ready-Made Meals": 0,
  "Ethnic and Specialty Foods": 0,
  "Culinary Specialties": 0
}"""

food_category_indices = []

for i in range(len(food_cat_guys)):
    # print(i)
    dictionary = json.loads(food_cat_guys[i])

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(food_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    food_category_indices.append(indices)

food_product_type_list = [making_product_type_list(food_category_indices[i], all_food_products) for i in range(len(food_cat_guys))]

In [ ]:
electronic_item_names['description'] = text_electronics
clothing_item_names['description'] = text_clothing
food_item_names['description'] = text_food

dicts = [json.loads(s) for s in metadata_electronic_jsons]

metadata_df = pd.DataFrame(dicts)

columns_to_add = metadata_df[['product_type', 'brand', 'design_focus', 'target_user', 'cost_tier', 'key_features']]

electronic_item_jazzed = pd.concat([electronic_item_names, columns_to_add], axis=1)

dicts = [json.loads(s) for s in metadata_electronic_jsons]

metadata_df = pd.DataFrame(dicts)

columns_to_add = metadata_df[['product_type', 'brand', 'design_focus', 'target_user', 'cost_tier', 'key_features']]

electronic_item_jazzed = pd.concat([electronic_item_names, columns_to_add], axis=1)

dicts = [json.loads(s) for s in metadata_clothing_jsons]

metadata_df = pd.DataFrame(dicts)

columns_to_add = metadata_df[['product_type', 'brand', 'design_focus', 'target_user', 'cost_tier', 'key_features']]

clothing_item_jazzed = pd.concat([clothing_item_names, columns_to_add], axis=1)

dicts = [json.loads(s) for s in metadata_food_jsons]

metadata_df = pd.DataFrame(dicts)

columns_to_add = metadata_df[['product_type', 'brand', 'dietary_considerations', 'flavor_profile', 'cost_tier', 'key_features']]

food_item_jazzed = pd.concat([food_item_names, columns_to_add], axis=1)



In [72]:
# 1. Loop through each JSON string in your list
# 2. Extract keys where value is 1
# 3. Join them into a single string (if multiple categories apply)
category_results = []

for json_str in clothing_cat_guys:
    # Parse the string into a dictionary
    data_dict = json.loads(json_str)

    # Get all keys that have a value of 1
    active_keys = [key for key, value in data_dict.items() if value == 1]

    # Join them with a comma (e.g., "Gaming, Computers and Accessories")
    # If no category is 1, it will result in an empty string ""
    category_results.append(", ".join(active_keys))

# Now you can add this list directly to your DataFrame
clothing_item_jazzed['categories'] = category_results

In [73]:
# 1. Loop through each JSON string in your list
# 2. Extract keys where value is 1
# 3. Join them into a single string (if multiple categories apply)
category_results = []

for json_str in electronic_cat_guys:
    # Parse the string into a dictionary
    data_dict = json.loads(json_str)

    # Get all keys that have a value of 1
    active_keys = [key for key, value in data_dict.items() if value == 1]

    # Join them with a comma (e.g., "Gaming, Computers and Accessories")
    # If no category is 1, it will result in an empty string ""
    category_results.append(", ".join(active_keys))

# Now you can add this list directly to your DataFrame
electronic_item_jazzed['categories'] = category_results

In [74]:
# 1. Loop through each JSON string in your list
# 2. Extract keys where value is 1
# 3. Join them into a single string (if multiple categories apply)
category_results = []

for json_str in food_cat_guys:
    # Parse the string into a dictionary
    data_dict = json.loads(json_str)

    # Get all keys that have a value of 1
    active_keys = [key for key, value in data_dict.items() if value == 1]

    # Join them with a comma (e.g., "Gaming, Computers and Accessories")
    # If no category is 1, it will result in an empty string ""
    category_results.append(", ".join(active_keys))

# Now you can add this list directly to your DataFrame
food_item_jazzed['categories'] = category_results

In [79]:

electronic_item_jazzed.to_csv('/content/electronic/electronic_items_categories.csv', index=False, encoding='utf-8-sig')
clothing_item_jazzed.to_csv('/content/clothing/clothing_items_categories.csv', index=False, encoding='utf-8-sig')
food_item_jazzed.to_csv('/content/food/food_items_categories.csv', index=False, encoding='utf-8-sig')

In [83]:

def session_info_provider(session_ID, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2


    item_ids = session_item[k][session_item[k]["session ID"] == session_ID]["item ID"].values

    # A list to store the titles of the items that were successfully found
    session_item_titles = []


    existing = []

    for item_id in item_ids:
        item_titles = item_names[k][item_names[k]["item ID"] == item_id]["titles"]
        if not item_titles.empty and item_titles.values[0] not in existing:
            session_item_titles.append(item_titles.values[0])

            existing.append(item_titles.values[0])

    session_item_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in session_item_titles]


    return session_item_ids, session_item_titles


In [80]:
session_to_electronic_bundles_series = electronic_session_bundle.groupby('session ID')['bundle ID'].apply(list)
session_to_clothing_bundles_series = clothing_session_bundle.groupby('session ID')['bundle ID'].apply(list)
session_to_food_bundles_series = food_session_bundle.groupby('session ID')['bundle ID'].apply(list)

In [84]:
original_electronic_session_items = [session_info_provider(i, 'electronic')[1] for i in range(893)]
original_clothing_session_items = [session_info_provider(i, 'clothing')[1] for i in range(968)]
original_food_session_items = [session_info_provider(i, 'food')[1] for i in range(914)]

In [85]:
llm4bear_electronic_session_items = copy.deepcopy(original_electronic_session_items)
llm4bear_clothing_session_items = copy.deepcopy(original_clothing_session_items)
llm4bear_food_session_items = copy.deepcopy(original_food_session_items)

In [87]:
for i in range(len(session_to_electronic_bundles_series)):
    bundle_ids = session_to_electronic_bundles_series[i]
    # print(bundle_ids)
    for bundle_id in bundle_ids:
        # print(bundle_id)
        bundle_items = llm4bear_electronic_bundles[bundle_id]
        # print(bundle_items)
        for j in bundle_items:
            # print(j)
            if j not in llm4bear_electronic_session_items[i]:
                llm4bear_electronic_session_items[i].append(j)


In [88]:
for i in range(len(session_to_clothing_bundles_series)):
    bundle_ids = session_to_clothing_bundles_series[i]
    # print(bundle_ids)
    for bundle_id in bundle_ids:
        # print(bundle_id)
        bundle_items = llm4bear_clothing_bundles[bundle_id]
        # print(bundle_items)
        for j in bundle_items:
            # print(j)
            if j not in llm4bear_clothing_session_items[i]:
                llm4bear_clothing_session_items[i].append(j)


In [89]:
for i in range(len(session_to_food_bundles_series)):
    bundle_ids = session_to_food_bundles_series[i]
    # print(bundle_ids)
    for bundle_id in bundle_ids:
        # print(bundle_id)
        bundle_items = llm4bear_food_bundles[bundle_id]
        # print(bundle_items)
        for j in bundle_items:
            # print(j)
            if j not in llm4bear_food_session_items[i]:
                llm4bear_food_session_items[i].append(j)


In [101]:
electronic_session_index = []
electronic_session_item_ID = []

for i in range(len(llm4bear_electronic_session_items)):
    for target_title in llm4bear_electronic_session_items[i]:
        electronic_session_index.append(i)
        item_id = electronic_metadata.loc[electronic_metadata['titles'] == target_title, 'item ID'].values[0]
        electronic_session_item_ID.append(item_id)

clothing_session_index = []
clothing_session_item_ID = []

for i in range(len(llm4bear_clothing_session_items)):
    for target_title in llm4bear_clothing_session_items[i]:
        clothing_session_index.append(i)
        item_id = clothing_metadata.loc[clothing_metadata['titles'] == target_title, 'item ID'].values[0]
        clothing_session_item_ID.append(item_id)

food_session_index = []
food_session_item_ID = []

for i in range(len(llm4bear_food_session_items)):
    for target_title in llm4bear_food_session_items[i]:
        food_session_index.append(i)
        item_id = food_metadata.loc[food_metadata['titles'] == target_title, 'item ID'].values[0]
        food_session_item_ID.append(item_id)

In [102]:
def session_item_maker(session_item_index, session_item_ID, domain):
    # 1. Create a dictionary with your column names and lists
    data = {
        'session ID': session_item_index,
        'item ID': session_item_ID
    }

    # 2. Convert to a DataFrame
    df = pd.DataFrame(data)

    os.makedirs(f"/content/{domain}", exist_ok=True)

    # 3. Save to CSV (index=False prevents an extra column of numbers)
    df.to_csv(f'{domain}/session_item.csv', index=False)

    print("CSV created successfully!")

In [103]:
session_item_maker(electronic_session_index, electronic_session_item_ID, "electronic")
session_item_maker(clothing_session_index, clothing_session_item_ID, "clothing")
session_item_maker(food_session_index, food_session_item_ID, "food")

CSV created successfully!
CSV created successfully!
CSV created successfully!


In [104]:
# Zip everything inside /content/ (excluding system folders)
!zip -r all_my_items.zip /content/electronic /content/clothing /content/food

# Download the big zip
from google.colab import files
files.download('all_my_items.zip')

  adding: content/electronic/ (stored 0%)
  adding: content/electronic/bundle_item.csv (deflated 61%)
  adding: content/electronic/bundle_intent.csv (deflated 71%)
  adding: content/electronic/.ipynb_checkpoints/ (stored 0%)
  adding: content/electronic/session_item.csv (deflated 60%)
  adding: content/electronic/electronic_items_categories.csv (deflated 72%)
  adding: content/clothing/ (stored 0%)
  adding: content/clothing/bundle_item.csv (deflated 61%)
  adding: content/clothing/bundle_intent.csv (deflated 68%)
  adding: content/clothing/session_item.csv (deflated 59%)
  adding: content/clothing/clothing_items_categories.csv (deflated 73%)
  adding: content/food/ (stored 0%)
  adding: content/food/bundle_item.csv (deflated 61%)
  adding: content/food/bundle_intent.csv (deflated 70%)
  adding: content/food/session_item.csv (deflated 59%)
  adding: content/food/food_items_categories.csv (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>